# Time series transformer prototype

See the repository README and reproduction guide for data requirements and experiment settings. Generated models and histories are written to `outputs/time_series_transformer_prototype/`.

**Experimental prototype:** the original notebook contains a TimeSeriesTransformer experiment, not a verified pretrained-model pipeline. Its full runtime has not been validated.


In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "research_paths.py").is_file()
)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
from research_paths import data_file, external_file, checkpoint_file, history_file, output_file


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from transformers import TimeSeriesTransformerForPrediction, TimeSeriesTransformerConfig
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==============================
# 1. Load and Preprocess Data
# ==============================

# Set seed for reproducibility
seed_value = 777
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)

# Load training dataset
path_train = data_file("spectra/training_data_long_ontime.csv")
data = pd.read_csv(path_train, dtype=float)

# Extract spectrum (1862 features) and target variable (Cu concentration)
spectrum = data.iloc[:, 18:1880].values
spectrum = np.clip(spectrum, None, 60000) / 60000  # Normalize to 0-1 range
concentration = data["Cu"].values  # Target variable

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    spectrum, concentration, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Expand dimensions for PyTorch (match expected shape: (batch_size, sequence_length, num_features))
X_train = np.expand_dims(X_train, axis=1)  # Shape: (batch_size, 1, 1862)
X_val = np.expand_dims(X_val, axis=1)

y_train = np.expand_dims(y_train, axis=-1)  # Shape: (batch_size, 1)
y_val = np.expand_dims(y_val, axis=-1)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

# ==============================
# 2. Define Model
# ==============================

# Define Transformer model configuration
config = TimeSeriesTransformerConfig(
    prediction_length=1,
    context_length=1,  # Keep this at 1 since we only have one time step
    input_size=1862,  # Number of features per time step
    num_time_features=1,  # Must include at least 1 time feature
    num_static_categorical_features=1,  # Must be >= 1
    num_static_real_features=1,  # Must be >= 1
    encoder_layers=4,
    decoder_layers=4,
    d_model=64,
    n_heads=4,
    activation="relu",
    lags_sequence=[0],  # Only use current time step (no lagged values)
)

# Initialize Transformer Model
model = TimeSeriesTransformerForPrediction(config)

# Modify output layer to a single regression output
model.model.decoder.prediction_head = nn.Linear(config.d_model, 1)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ==============================
# 3. Create Data Loaders
# ==============================

# Create observed mask (all ones, indicating no missing values)
observed_mask_train = torch.ones_like(X_train)
observed_mask_val = torch.ones_like(X_val)

# Create dummy past time features (ones, not zeros) - IMPORTANT: Must not be None
past_time_features_train = torch.ones(
    (X_train.shape[0], X_train.shape[1], 1), dtype=torch.float32
)  # Simple time feature
past_time_features_val = torch.ones((X_val.shape[0], X_val.shape[1], 1), dtype=torch.float32)

# Create static categorical & real features with correct shapes
static_categorical_features_train = torch.zeros(
    (X_train.shape[0], 1), dtype=torch.long
)  # Shape: (batch_size, 1) and long dtype for categorical
static_real_features_train = torch.zeros(
    (X_train.shape[0], 1), dtype=torch.float32
)  # Shape: (batch_size, 1)
static_categorical_features_val = torch.zeros((X_val.shape[0], 1), dtype=torch.long)
static_real_features_val = torch.zeros((X_val.shape[0], 1), dtype=torch.float32)

# Define dataset and dataloaders
train_dataset = TensorDataset(
    X_train,
    past_time_features_train,
    observed_mask_train,
    static_categorical_features_train,
    static_real_features_train,
    y_train,
)
val_dataset = TensorDataset(
    X_val,
    past_time_features_val,
    observed_mask_val,
    static_categorical_features_val,
    static_real_features_val,
    y_val,
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# ==============================
# 4. Train the Model
# ==============================

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for inputs, past_features, mask, cat_features, real_features, targets in train_loader:
        inputs, past_features, mask, cat_features, real_features, targets = (
            inputs.to(device),
            past_features.to(device),
            mask.to(device),
            cat_features.to(device),
            real_features.to(device),
            targets.to(device),
        )

        optimizer.zero_grad()

        # Model Forward Pass - Get the correct output field
        outputs = model(
            past_values=inputs,
            past_time_features=past_features,
            past_observed_mask=mask,
            static_categorical_features=cat_features,
            static_real_features=real_features,
        )

        # The output is in the 'last_decoder_hidden_state' or 'decoder_hidden_states' field
        # Let's extract the prediction from the model output
        preds = outputs.last_hidden_state[:, -1].unsqueeze(
            1
        )  # Take last token and add batch dimension

        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation Step
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, past_features, mask, cat_features, real_features, targets in val_loader:
            inputs, past_features, mask, cat_features, real_features, targets = (
                inputs.to(device),
                past_features.to(device),
                mask.to(device),
                cat_features.to(device),
                real_features.to(device),
                targets.to(device),
            )

            outputs = model(
                past_values=inputs,
                past_time_features=past_features,
                past_observed_mask=mask,
                static_categorical_features=cat_features,
                static_real_features=real_features,
            )

            # Extract predictions same way as in training
            preds = outputs.last_hidden_state[:, -1].unsqueeze(1)

            loss = criterion(preds, targets)
            val_loss += loss.item() * inputs.size(0)

    val_loss /= len(val_loader.dataset)
    print(
        f"Epoch {epoch+1}/{num_epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}"
    )

# ==============================
# 5. Save Model
# ==============================

torch.save(
    model.state_dict(),
    output_file("transformer_spectral_regression.pth", "time_series_transformer_prototype"),
)
